Note: LTV excluded from gold_default_by_segment. Median imputation on Day 2 created a 15,203-row spike at the median value (confirmed via frequency check), distorting LTV-based segmentation. Documented as a known data limitation rather than included in the trusted layer.

In [0]:
%sql
CREATE OR REPLACE TABLE loan_default_workspace.default.gold_regional_breakdown AS
SELECT
    Region,
    COUNT(*) AS total_loans,
    COUNT(CASE WHEN Status = 1 THEN 1 END) AS defaulted_loans,
    ROUND(COUNT(CASE WHEN Status = 1 THEN 1 END) / Count(Status) * 100, 2) AS default_rate_pct,
    SUM(CASE WHEN Status = 1 THEN loan_amount ELSE 0 END) AS value_at_risk
FROM loan_default_workspace.default.loans_silver
GROUP BY Region
ORDER BY value_at_risk DESC

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE loan_default_workspace.default.gold_risk_summary AS
SELECT 
    Region,
    COUNT(*) AS loan_count,
    CASE 
        WHEN dtir1 < 30 THEN 'Under 30' 
        WHEN dtir1 >= 30 THEN 'Over 30' 
    END AS dti_bucket,
  ROUND(COUNT(CASE WHEN Status = 1 THEN 1 END)/ Count(Status)*100.0,2) AS default_rate_pct 
FROM loan_default_workspace.default.loans_silver 
GROUP BY Region, CASE WHEN dtir1 < 30 THEN 'Under 30' WHEN dtir1 >= 30 THEN 'Over 30' END
order by default_rate_pct desc


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    'Credit Band' AS segment_type,           -- a fixed text label, same on every row in this block
    CASE 
        WHEN Credit_Score < 600 THEN 'Low'
        WHEN Credit_Score BETWEEN 600 AND 700 THEN 'Medium'
        ELSE 'High'
    END AS segment_value,
    COUNT(*) AS loan_count,
  ROUND(COUNT(CASE WHEN Status = 1 THEN 1 END)/ Count(Status)*100.0,2) AS default_rate_pct 
FROM loan_default_workspace.default.loans_silver
GROUP BY     CASE 
        WHEN Credit_Score < 600 THEN 'Low'
        WHEN Credit_Score BETWEEN 600 AND 700 THEN 'Medium'
        ELSE 'High'
    END

segment_type,segment_value,loan_count,default_rate_pct
Credit Band,High,73877,24.73
Credit Band,Low,37190,24.63
Credit Band,Medium,37603,24.49


In [0]:
%sql
CREATE OR REPLACE TABLE loan_default_workspace.default.gold_default_by_segment AS
SELECT
    'Credit Band' AS segment_type,
    CASE WHEN Credit_Score < 600 THEN 'Low'
         WHEN Credit_Score BETWEEN 600 AND 700 THEN 'Medium'
         ELSE 'High' END AS segment_value,
    COUNT(*) AS loan_count,
    ROUND(SUM(CASE WHEN Status = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loan_default_workspace.default.loans_silver
GROUP BY 2

UNION ALL

SELECT
    'DTI Band' AS segment_type,
  CASE 
    WHEN dtir1 < 30 THEN 'Low' 
    WHEN dtir1 BETWEEN 30 AND 45 THEN 'Medium' 
    WHEN dtir1 > 45 THEN 'High' 
end AS dti_bucket, 
 COUNT(*) AS loan_count, 
    ROUND(SUM(CASE WHEN Status = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loan_default_workspace.default.loans_silver
GROUP BY 2

num_affected_rows,num_inserted_rows
